# credit

> Discovery credit and scouting bandits: can rewarding the information an agent finds make some agents explore for the team?

The first agent to harvest at a patch since it last moved is its **discoverer**; each later harvest by another agent there pays the discoverer `beta` x that harvest as a learning signal (not welfare). A `ScoutBandit` learns between exploiting and scouting (going to its least-known location); credit is paid into the mode that made the discovery, so with `beta > 0` scouting can pay even when it harvests little. Plan: `specs/multiagent/README.md` (Phase 8, group vs individual reward).

In [ ]:
#| default_exp credit

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import defaultdict, deque

import numpy as np

from explRL.harness_agents import CommBandit
from explRL.harness_runs import benchWorld, harnessMetrics, makeChannel, worldTime
from explRL.multi import runTeam


class Credit:
    """Discovery credit. The first agent to harvest at a patch since the patch last moved is its discoverer;
    every later harvest by another agent there pays the discoverer `beta` x that harvest, as a learning signal
    delivered at the discoverer's next event. Credit is a transfer of information value, not harvest: welfare
    does not count it."""
    def __init__(self, n, beta=1.0):
        self.beta, self.first = float(beta), {}
        self.inbox = [deque() for _ in range(n)]
        self.stats = defaultdict(lambda: dict(discovered=0, credited=0.0))

    def stay(self, world, i, loc, harvest, mode):
        "Record agent i's stay (true harvest) at loc, made in `mode`; returns the discoverer paid, or None."
        if harvest <= 0 or world.patchAt(loc) is None:
            return None
        key = (int(loc), world.nextMove)              # one discovery per patch site and move period
        if key not in self.first:
            self.first[key] = (i, mode)
            self.stats[i]['discovered'] += 1
            return None
        d, m = self.first[key]
        if d == i or self.beta == 0:
            return None
        c = self.beta * float(harvest)
        self.inbox[d].append((m, c))
        self.stats[d]['credited'] += c
        return d

    def take(self, i):
        out = list(self.inbox[i])
        self.inbox[i].clear()
        return out


class ScoutBandit(CommBandit):
    """`CommBandit` that learns, by UCB over two modes, whether to exploit (the `CommBandit` choice, mode 0) or
    scout (mode 1: go to its least-known location not believed bad, nearest first, and stay there once). A mode's
    value is its reward plus the discovery credit it earned, per second spent in it. Each mode is tried `warm`
    times before the UCB choice, so a few unlucky early events (a danger stay) cannot lock an agent into one mode."""
    def __init__(self, channel=None, credit=None, w=1.0, K=None, cUCB=1.0, H=10.0, cMode=0.2, warm=10):
        super().__init__(channel, w, K, cUCB, H)
        self.credit, self.cMode, self.warm = credit, float(cMode), int(warm)
        self.modeS, self.modeT, self.nMode = np.zeros(2), np.zeros(2), np.zeros(2, int)
        self.mode, self._scoutTo = 0, None

    def event(self, world, t):
        self._takeCredit()
        return super().event(world, t)

    def _takeCredit(self):
        "Add the discovery credit delivered since the last event to the mode that made each discovery."
        for m, c in (self.credit.take(self.id) if self.credit is not None else ()):
            self.modeS[m] += c

    def _pickMode(self):
        if (self.nMode < self.warm).any():
            return int(np.argmin(self.nMode))
        rate = np.where(self.modeT > 0, self.modeS / np.maximum(self.modeT, 1e-12), 0.0)
        return int(np.argmax(rate + self.cMode * np.sqrt(np.log1p(self.nMode.sum()) / self.nMode)))

    def act(self, obs):
        if self._scoutTo is not None and self._scoutTo == obs['location']:
            self.mode, self._scoutTo = 1, None        # a scout trip ends with one stay at its target
            self.nMode[1] += 1
            return obs['location']
        self._scoutTo = None
        self.mode = self._pickMode()
        self.nMode[self.mode] += 1
        if self.mode == 0:
            return super().act(obs)
        T = obs['travel']
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in T]
        mean, n = self._stats(obs)
        ok = np.flatnonzero((n == 0) | (mean >= 0))   # skip places it already believes are bad
        ok = ok if len(ok) else np.arange(len(T))
        j = int(ok[np.lexsort((T[ok], n[ok]))[0]])    # least known, then nearest
        self._scoutTo = j if j != obs['location'] else None
        return j

    def learn(self, obs, target, reward, info):
        super().learn(obs, target, reward, info)
        self.modeS[self.mode] += reward
        self.modeT[self.mode] += info['dur']
        if self.credit is not None and info['kind'] == 'stay':
            self.credit.stay(self._world, self.id, obs['location'], info['harvest'], self.mode)


def runCredit(cfg, N, seed, beta=1.0, cMode=0.2, warm=10):
    "Like `runHarness` for a team of `ScoutBandit`s; `beta = None` runs plain `CommBandit`s (no scouting, no credit)."
    world = benchWorld(**cfg.world)
    world.reset(seed, N)
    channel = makeChannel(cfg, world.L, N, seed)
    credit = Credit(N, beta) if beta is not None else None
    if beta is None:
        members = [CommBandit(channel, cfg.w, cfg.K, cfg.cUCB, cfg.H) for _ in range(N)]
    else:
        members = [ScoutBandit(channel, credit, cfg.w, cfg.K, cfg.cUCB, cfg.H, cMode, warm) for _ in range(N)]
    T = worldTime(N, cfg)
    hist = runTeam(members, world, T)
    m = harnessMetrics(cfg, world, hist, channel, members, N, T)
    if beta is not None:
        share = [float(x.nMode[1] / max(x.nMode.sum(), 1)) for x in members]
        m.update(beta=beta, cMode=cMode, warm=warm, scoutShare=share, discovered=sum(s['discovered'] for s in credit.stats.values()),
                 credited=float(sum(s['credited'] for s in credit.stats.values())))
    return world, hist, credit, m

## Tests

In [ ]:
import numpy as np
from collections import deque
from fastcore.test import test_eq, test_close
from explRL.credit import Credit, ScoutBandit, runCredit
from explRL.harness import Blackboard
from explRL.harness_runs import HarnessConfig, benchWorld

# Credit: first harvester at a patch is its discoverer; later harvests by others pay it beta x harvest;
# its own later harvests and empty or non-patch stays pay nothing; a patch move starts a new discovery
w = benchWorld(nPatches=1, nDanger=0, intake=(2.0,), patchSites=[0], dangerSites=[], Tmove=40.0)
w.reset(0, 3)
cr = Credit(3, beta=0.5)
test_eq(cr.stay(w, 0, 0, 0.8, 1), None)               # agent 0 discovers, in scout mode
test_eq(cr.stay(w, 0, 0, 0.8, 0), None)               # its own later harvest: no credit
test_eq(cr.stay(w, 1, 0, 0.8, 0), 0)                  # agent 1 harvests: pays agent 0
test_eq(cr.stay(w, 2, 5, 0.8, 0), None)               # not a patch
test_eq(cr.stay(w, 2, 0, 0.0, 0), None)               # nothing harvested
test_eq(cr.take(0), [(1, 0.4)]); test_eq(cr.take(0), [])
test_eq(cr.stats[0]['discovered'], 1); test_close(cr.stats[0]['credited'], 0.4)
w.advance(40.0)                                        # the patch moves: a new discovery is possible
new = int(w.site[0])
test_eq(cr.stay(w, 2, new, 0.8, 0), None); test_eq(cr.stats[2]['discovered'], 1)
c0 = Credit(3, beta=0.0); c0.stay(w, 0, new, 0.8, 0)
test_eq(c0.stay(w, 1, new, 0.8, 0), None); test_eq(c0.take(0), [])

# ScoutBandit scout mode: goes to the least-known location (nearest first), then stays there once
sb = ScoutBandit(Blackboard(4, n=1), Credit(1), w=1.0)
wd = benchWorld(L=4, nPatches=1, nDanger=0, intake=(1.0,), patchSites=[0], dangerSites=[]); wd.reset(0, 1)
sb.id, sb._world, sb._t = 0, wd, 0.0
sb.rates = [deque([1.0]), deque([1.0, 1.0]), deque(), deque([-2.6])]
sb.nMode[:] = (20, 20); sb.modeS[:] = (0.0, 9.0); sb.modeT[:] = (10.0, 10.0)    # scouting has paid more
test_eq(sb.act({'location': 0, 'travel': wd.layout.T[0]}), 2); test_eq(sb.mode, 1)
test_eq(sb.act({'location': 2, 'travel': wd.layout.T[2]}), 2); test_eq(sb.mode, 1)
sb.rates[2].append(0.5); sb.rates[1].append(1.0)                    # now 3 (n = 1) is least known, but bad
test_eq(sb.act({'location': 1, 'travel': wd.layout.T[1]}), 0)       # so from 1 the scout picks 0 (n = 1, nearest, lower index)
# credit received is added to the mode that made the discovery
sb.credit.inbox[0].append((1, 0.7)); before = sb.modeS.copy()
sb._takeCredit()
test_close(sb.modeS - before, [0.0, 0.7]); test_eq(sb.credit.take(0), [])

# runs: deterministic; beta = None is plain CommBandit (same history as runHarness); welfare excludes credit
cfg = HarnessConfig(channel='blackboard', T=120.0)
a, b = runCredit(cfg, 8, 0, beta=1.0), runCredit(cfg, 8, 0, beta=1.0)
test_eq(a[1].records, b[1].records); test_eq(a[3]['credited'], b[3]['credited'])
assert a[3]['discovered'] > 0 and len(a[3]['scoutShare']) == 8
from explRL.harness_runs import runHarness
test_eq(runCredit(cfg, 8, 0, beta=None)[1].records, runHarness(cfg, 8, 0)[1].records)
test_close(a[3]['welfare'], a[3]['harvest'] - a[3]['dangerCost'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()